# TuringBench DeBERTa-v3-large with LoRA (Kaggle)

This notebook uses **4-bit LoRA** to fine-tune `microsoft/deberta-v3-large` on TuringBench.
It is much more stable and memory-efficient than full fine-tuning on Kaggle P100.

Before you run: add `HF_TOKEN` to Kaggle Secrets (Add-ons → Secrets).

In [ ]:
# Install PyTorch 2.4.1 for P100 (CUDA 11.8) and training dependencies
!pip install -q torch==2.4.1+cu118 torchvision==0.19.1+cu118 torchaudio==2.4.1+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets accelerate scikit-learn pandas joblib huggingface-hub peft bitsandbytes

In [ ]:
import os
import shutil

# Read HuggingFace token from Kaggle Secrets.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets')
except Exception as e:
    print('Could not load HF_TOKEN from Kaggle Secrets:', e)
    print('Set HF_TOKEN via Add-ons -> Secrets.')

repo_url = 'https://github.com/vedangvatsa/ai-detection-at-scale.git'
repo_dir = '/kaggle/working/ai-detection-at-scale'
# Force a fresh clone so we always use the latest code.
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)
!git clone $repo_url $repo_dir
print('Repo ready at', repo_dir)

In [ ]:
import sys
sys.path.insert(0, os.path.join(repo_dir, 'scripts'))

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.metrics import roc_auc_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'microsoft/deberta-v3-large'
HUB_MODEL_ID = 'vedangvatsa123/vedang-turingbench-deberta-v3-large-lora'
OUTPUT_DIR = '/kaggle/working/models/turingbench_deberta_v3_large_lora'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Load TuringBench dataset
dataset = load_dataset('turingbench/TuringBench', revision='refs/convert/parquet')

def preprocess_split(split, max_samples=None):
    df = split.to_pandas()
    df['label'] = df['label'].apply(lambda x: 0 if str(x).lower() == 'human' else 1)
    df['text'] = df['Generation'].astype(str)
    df = df[df['text'].str.len() >= 20].reset_index(drop=True)
    if max_samples:
        df = df.sample(min(max_samples, len(df)), random_state=42).reset_index(drop=True)
    return df[['text', 'label']]

train_df = preprocess_split(dataset['train'])
val_df = preprocess_split(dataset['validation'])

print('Train rows:', len(train_df), train_df['label'].value_counts().to_dict())
print('Val rows:', len(val_df), val_df['label'].value_counts().to_dict())

In [ ]:
MAX_LENGTH = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=MAX_LENGTH, padding=False)

from datasets import Dataset
train_ds = Dataset.from_pandas(train_df).map(tokenize_function, batched=True, remove_columns=['text'])
val_ds = Dataset.from_pandas(val_df).map(tokenize_function, batched=True, remove_columns=['text'])

train_ds.set_format('torch')
val_ds.set_format('torch')

In [ ]:
# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'human', 1: 'ai'},
    label2id={'human': 0, 'ai': 1},
    quantization_config=bnb_config,
    device_map='auto',
)

# Apply LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['query_proj', 'key_proj', 'value_proj', 'dense'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_CLS,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
can_push = os.environ.get('HF_TOKEN') is not None
if not can_push:
    print('WARNING: HF_TOKEN not set. Model will be saved locally but not pushed to Hub.')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_steps=500,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='auc',
    greater_is_better=True,
    report_to=['none'],
    seed=42,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    max_grad_norm=1.0,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    push_to_hub=can_push,
    hub_model_id=HUB_MODEL_ID if can_push else None,
    hub_strategy='checkpoint' if can_push else 'every_save',
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = np.argmax(logits, axis=-1)
    return {
        'auc': roc_auc_score(labels, probs),
        'accuracy': accuracy_score(labels, preds),
    }

data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Save final model
model.save_pretrained(os.path.join(OUTPUT_DIR, 'final_model'))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, 'final_model'))
print('Training complete. Model saved to', OUTPUT_DIR)